# GridWorld (Short Explanation)

**GridWorld** is a simple Reinforcement Learning environment where an **agent moves inside a grid to reach a goal**.

## Environment
- The agent starts at a **starting position**.
- The grid also contains **obstacles** and a **goal**.
- The agent can move in **four directions**:
  - Up
  - Down
  - Left
  - Right

Each cell in the grid is converted into a **state number** so the learning algorithm can work with it.

## Reward System
- **Goal reached → +10 reward**
- **Obstacle hit → -10 reward**
- **Normal move → -1 reward**
- **Moving outside grid → -1 reward**

## Learning Process
The agent learns using a **Q-table**, which stores the value of taking each action in every state.

To choose actions, the agent uses the **epsilon-greedy strategy**:
- **Exploration:** choose a random action to discover new paths.
- **Exploitation:** choose the best action based on the Q-table.

After many episodes, the agent learns the **optimal path to reach the goal while avoiding obstacles**.

In [1]:
import numpy as np
class GridWorld():
  def __init__(self,size):
    self.size=size
    self.start=(0,0)
    self.obstacles=[(1,1),(2,3),(3,0)]
    self.goal=(size-1,size-1)
    self.agent_pos=self.start

  def position_to_state(self,pos): #This function converts the agent’s position in the grid into a single state number.
    row,col=pos
    return row * self.size + col
  def reset(self): #This function starts a new episode by putting the agent back at the starting position.
    self.agent_pos=self.start
    return self.position_to_state(self.agent_pos)

  def step(self,action):   #action which the agent will perform in the grid to reach the goal
    actions={
        0: (-1,0),  #up
        1: (1,0),   #down
        2: (0,-1),  #left
        3: (0,1)    #right
    }
    #get current position of agent
    row,col=self.agent_pos
    #get movement change
    delta_row,delta_col=actions[action]  #get the values from key of actions  example action = 3   # Right  , actions[3]=right=(0,1)
    new_row=row+delta_row   #agent gets new direction to move
    new_col=col+delta_col

    #check boundaries so that the agent dont cross the grid
    if new_row < 0 or new_row>=self.size or new_col<0 or new_col>=self.size:    #if row and col becomes -ve or greater than grid size ,the reawrd will be negetive
      reward=-1
      done=False
      return self.position_to_state(self.agent_pos),reward,done
    new_pos=(new_row,new_col)

    #obstacle hit case
    if new_pos in self.obstacles:
      self.agent_pos = new_pos
      reward=-10
      done=True
      return self.position_to_state(self.agent_pos),reward,done
    #Goa move
    elif new_pos ==self.goal:
      self.agent_pos=new_pos
      reward=10
      done=True
    #NORMAL move
    else:
      self.agent_pos=new_pos
      reward=-1
      done=False

    return self.position_to_state(self.agent_pos),reward,done

def q_table(env):


  states=env.size*env.size
  no_of_actions=4

  q=np.zeros((states,no_of_actions))
  return q
def choose_action(state,Q,epsilon):
  if np.random.rand()<epsilon:
    action=np.random.randint(0,4)
  else:
    action=np.argmax(Q[state])

  return action

# Q-Learning Training Process (GridWorld)

This part of the code **trains the agent using Q-learning** so it can learn the best path to the goal.

## Training Parameters
- **Episodes = 5000** → Number of times the agent plays the game to learn.
- **Alpha (0.1)** → Learning rate (how quickly the agent updates knowledge).
- **Gamma (0.9)** → Discount factor (importance of future rewards).
- **Epsilon (0.1)** → Exploration rate (probability of choosing a random action).

## Environment Setup
- A **5×5 GridWorld environment** is created.
- A **Q-table** is initialized to store values for each **state–action pair**.

## Training Loop
For each episode:
1. The environment is **reset**, and the agent returns to the start.
2. The agent continues moving **until it reaches the goal or hits an obstacle**.

## Action Selection
The agent chooses an action using **epsilon-greedy strategy**:
- **Exploration:** choose random action.
- **Exploitation:** choose the best action from the Q-table.

## Environment Interaction
After taking an action:
- The environment returns:
  - **Next state**
  - **Reward**
  - **Done flag** (episode finished or not)

## Q-Value Update
The agent updates the Q-table using the **Q-learning update rule**, which adjusts the value of the current state-action pair based on:
- current reward
- best possible future reward

This allows the agent to **gradually learn better decisions**.

## Training Result
After all episodes finish:
- The **trained Q-table is saved** to a file.

This Q-table represents the **learned policy**, which can later be used to make the agent automatically follow the **optimal path to the goal**.

In [2]:
episodes=5000
alpha=0.1 #learning rate
gamma=0.9 #discount factor
epsilon=0.1 #exploration rate

env=GridWorld(5)
Q=q_table(env)
for episode in range(episodes):
  state=env.reset()
  done=False
  while not done:  #till the agent reaches goal or dead by obstacle
    action=choose_action(state,Q,epsilon)
    next_state,reward,done=env.step(action)  #next action of the agent
    best_next_action=np.max(Q[next_state])
    #reinforcement learning formula
    Q[state,action]=Q[state,action]+ alpha * (reward + gamma * best_next_action- Q[state,action] )
    state=next_state
np.save("q_table.npy", Q)


In [3]:
#print(Q)

# Policy Visualization (GridWorld)

This function displays the **learned policy of the agent** using arrows on the grid.

## Arrow Representation
Each action is represented by a direction arrow:
- **↑** → Move Up
- **↓** → Move Down
- **←** → Move Left
- **→** → Move Right

## Grid Iteration
The function goes through **every cell of the grid** row by row and column by column.

## Special Cells
Some cells show fixed symbols:
- **G** → Goal position
- **X** → Obstacle

These cells do not show arrows.

## Best Action Selection
For normal cells:
1. The position is converted into a **state number**.
2. The function looks at the **Q-table**.
3. It finds the **action with the highest Q-value**.

This action represents the **best move learned by the agent**.

## Visualization
The best action for each state is printed as an **arrow**, creating a grid that shows the **optimal direction the agent should move from every position**.

## Result
The printed grid becomes a **policy map**, helping us visually understand the **path the agent learned to reach the goal while avoiding obstacles**.

In [4]:
def show(Q, env):

    arrows = {
        0: "↑",
        1: "↓",
        2: "←",
        3: "→"
    }

    for row in range(env.size):
        line = ""

        for col in range(env.size):
            pos = (row, col)

            if pos == env.goal:
                line += " G "
            elif pos in env.obstacles:
                line += " X "
            else:
                state = env.position_to_state(pos)
                best_action = np.argmax(Q[state])
                line += f" {arrows[best_action]} "

        print(line)

In [5]:
show(Q,env)

 →  →  ↓  ↓  ↓ 
 ↓  X  ↓  →  ↓ 
 →  ↓  ↓  X  ↓ 
 X  →  ↓  →  ↓ 
 →  →  →  →  G 


# Grid Rendering (Environment Visualization)

This function prints the **current state of the GridWorld** so we can see where the agent is located.

## Grid Traversal
The function goes through **each row and column** of the grid to check what exists in every cell.

## Symbols Used
Different symbols represent different objects in the environment:

- **A** → Agent's current position  
- **O** → Obstacle  
- **G** → Goal  
- **.** → Empty cell  

## Position Check
For every grid cell:
1. If the cell contains the **agent**, it prints **A**.
2. If the cell contains an **obstacle**, it prints **O**.
3. If the cell contains the **goal**, it prints **G**.
4. Otherwise, it prints **.** for an empty space.

## Visualization
Each row is printed as a line, forming a **grid-like structure**.

Example output:


In [6]:
import time  #for printing grid after updation

def render(env):
  for row in range(env.size):
    line=""
    for col in range(env.size):
      #define position
      pos=(row,col)
      if pos==env.agent_pos:
        line+="A"
      elif pos in env.obstacles:
        line+="O"
      elif pos==env.goal:
        line+="G"
      else:
        line+="."
    print(line)
  print("\n")


# Agent Simulation (Watch Trained Agent)

This function allows us to **watch how the trained agent moves in the GridWorld using the learned Q-table**.

## Initialization
- The environment is **reset**, placing the agent at the starting position.
- `done` keeps track of whether the episode has finished.
- `running` controls the simulation loop.

## Screen Refresh
Before each step, the **terminal screen is cleared** so the grid updates like an animation.

## Grid Display
The current grid is printed using the **render function**, showing:
- **A** → Agent
- **O** → Obstacle
- **G** → Goal
- **.** → Empty cell

## User Controls
The user can control the simulation with commands:

- **p (play)** → Agent moves automatically until it reaches the goal.
- **s (step)** → Agent moves **one step at a time**.
- **q (quit)** → Stop the simulation.

## Agent Decision
The agent chooses the **best action from the Q-table** by selecting the action with the highest value.

## Automatic Play
When **play mode** is selected:
- The agent repeatedly takes the best action.
- The grid updates every **0.5 seconds** to show movement.

## End Condition
The simulation stops when:
- The agent **reaches the goal**, or
- The user **quits the program**.

## Purpose
This function helps **visualize the learned behavior of the agent** and confirms that the training successfully found the **optimal path to the goal**.

In [7]:
import os
import time
import numpy as np

def watch_agent(env, Q):

    state = env.reset()
    done = False
    running = True

    while running:

        os.system('cls' if os.name == 'nt' else 'clear')

        render(env)

        if done:
            print("Goal reached!")
            break

        print("\nControls:")
        print("p = play")
        print("s = step")
        print("q = quit")

        cmd = input("Enter command: ")

        if cmd == "q":
            running = False

        elif cmd == "s":
            action = np.argmax(Q[state])
            next_state, reward, done = env.step(action)
            state = next_state

        elif cmd == "p":

            while not done:

                os.system('cls' if os.name == 'nt' else 'clear')
                render(env)

                action = np.argmax(Q[state])
                next_state, reward, done = env.step(action)
                state = next_state

                time.sleep(0.5)

            print("Goal reached!")
            break

In [8]:
watch_agent(env,Q)

A....
.O...
...O.
O....
....G



Controls:
p = play
s = step
q = quit
Enter command: p
A....
.O...
...O.
O....
....G


.A...
.O...
...O.
O....
....G


..A..
.O...
...O.
O....
....G


.....
.OA..
...O.
O....
....G


.....
.O...
..AO.
O....
....G


.....
.O...
...O.
O.A..
....G


.....
.O...
...O.
O....
..A.G


.....
.O...
...O.
O....
...AG


Goal reached!
